In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [2]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9007::MY62310119::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [3]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [4]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99775900E+00', '+5.48400000E-03']
Default Signals Set
[128]


In [5]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99735100E+00', '+1.29310000E-02']


In [6]:
#Turn on CLK 
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY53270560, B.01.70


In [7]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [8]:
#Turn on Analog PS
#PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
#print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

In [9]:
#Create Checker pattern for test. Make sure calib supplys are off.
rows, cols = 296, 1156
checkerboard = np.fromfunction(lambda i, j: (i + j) % 2, (rows, cols), dtype=int)
# Step 2: Create zero array of full shape
full_array = np.zeros(shape=(320, 1156),dtype=np.uint8) #320 is the padded list.
#full_array[0:8,:] = 1
#full_array[:,1155] = 1
# Step 3: Insert checkerboard into left part (first 296 columns)
#full_array[:rows, :] = checkerboard
#full_array[:rows, :] = 1
#full_array[4:8,:] = 1
#full_array[0:8,:] = 0
#full_array[6:8,:] = 1

# Random positions
num_ones = 1156*150
ones_positions = np.random.choice(rows * cols, num_ones, replace=False)
full_array.flat[ones_positions] = 1
full_array = full_array.flatten(order='F') #Across Column


In [10]:
#Call Write SRAM here
write_data = full_array.tolist()  #Remember this is in bits. Will be converted to bytes inside
ret_data = Utils.Write_SRAM(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [11]:
arr = write_data
u8_array = []
for i in range(0, len(arr), 8):
    byte_bits = arr[i:i+8]
    value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
    u8_array.append(value)

print(u8_array)

[69, 117, 145, 131, 175, 41, 123, 63, 149, 253, 212, 137, 113, 203, 137, 219, 1, 71, 112, 193, 185, 163, 153, 241, 65, 68, 124, 33, 21, 175, 204, 184, 141, 53, 67, 240, 171, 0, 0, 0, 213, 86, 176, 20, 185, 16, 247, 108, 251, 205, 179, 124, 43, 182, 184, 39, 193, 35, 82, 221, 160, 242, 149, 211, 48, 227, 244, 103, 208, 136, 62, 183, 51, 2, 138, 179, 130, 0, 0, 0, 62, 219, 43, 231, 114, 164, 161, 207, 80, 192, 176, 252, 101, 190, 69, 133, 179, 59, 175, 166, 190, 58, 190, 25, 193, 141, 114, 105, 7, 10, 176, 215, 188, 181, 98, 91, 179, 0, 0, 0, 237, 96, 82, 27, 144, 74, 39, 100, 223, 40, 197, 176, 93, 132, 255, 171, 56, 252, 249, 203, 233, 190, 60, 176, 13, 34, 63, 143, 206, 107, 48, 196, 186, 76, 163, 154, 87, 0, 0, 0, 236, 0, 211, 215, 110, 100, 180, 101, 202, 179, 150, 55, 196, 64, 17, 183, 128, 254, 94, 19, 141, 191, 161, 26, 131, 139, 229, 202, 105, 232, 62, 70, 44, 123, 198, 163, 142, 0, 0, 0, 3, 154, 166, 204, 228, 9, 101, 145, 196, 43, 171, 240, 233, 132, 221, 86, 66, 45, 43, 58, 1

In [12]:
#Set IO to default and exit   
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data[0])

Default Signals Set
128


In [13]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 5
Received Data Size: 46240
All Data Received
46240
[69, 117, 145, 131, 175, 41, 123, 63, 149, 253, 212, 137, 113, 203, 137, 219, 1, 71, 112, 193, 185, 163, 153, 241, 65, 68, 124, 33, 21, 175, 204, 184, 141, 53, 67, 240, 171, 0, 0, 0, 213, 86, 176, 20, 185, 16, 247, 108, 251, 205, 179, 124, 43, 182, 184, 39, 193, 35, 82, 221, 160, 242, 149, 211, 48, 227, 244, 103, 208, 136, 62, 183, 51, 2, 138, 179, 130, 0, 0, 0, 62, 219, 43, 231, 114, 164, 161, 207, 80, 192, 176, 252, 101, 190, 69, 133, 179, 59, 175, 166, 190, 58, 190, 25, 193, 141, 114, 105, 7, 10, 176, 215, 188, 181, 98, 91, 179, 0, 0, 0, 237, 96, 82, 27, 144, 74, 39, 100, 223, 40, 197, 176, 93, 132, 255, 171, 56, 252, 249, 203, 233, 190, 60, 176, 13, 34, 63, 143, 206, 107, 48, 196, 186, 76, 163, 154, 87, 0, 0, 0, 236, 0, 211, 215, 110, 100, 180, 101, 202, 179, 150, 55, 196, 64, 17, 183, 128, 254, 94, 19, 141, 191, 161, 26, 131, 139, 229, 202, 105, 232, 62, 70, 44, 123

In [14]:
SRAM_DATA_Filtered = [ i for _,i in enumerate(SRAM_DATA) if (_ % 40) < 37]
u8_array_Filtered = [ i for _,i in enumerate(u8_array) if (_ % 40) < 37]
print((u8_array_Filtered))
print((SRAM_DATA_Filtered))
if SRAM_DATA_Filtered == u8_array_Filtered:
    print("Lists are equal")
else:
    print("Lists are not equal")

arr = np.array(SRAM_DATA_Filtered)
unique_elements = np.unique(arr)
#print(unique_elements)

[69, 117, 145, 131, 175, 41, 123, 63, 149, 253, 212, 137, 113, 203, 137, 219, 1, 71, 112, 193, 185, 163, 153, 241, 65, 68, 124, 33, 21, 175, 204, 184, 141, 53, 67, 240, 171, 213, 86, 176, 20, 185, 16, 247, 108, 251, 205, 179, 124, 43, 182, 184, 39, 193, 35, 82, 221, 160, 242, 149, 211, 48, 227, 244, 103, 208, 136, 62, 183, 51, 2, 138, 179, 130, 62, 219, 43, 231, 114, 164, 161, 207, 80, 192, 176, 252, 101, 190, 69, 133, 179, 59, 175, 166, 190, 58, 190, 25, 193, 141, 114, 105, 7, 10, 176, 215, 188, 181, 98, 91, 179, 237, 96, 82, 27, 144, 74, 39, 100, 223, 40, 197, 176, 93, 132, 255, 171, 56, 252, 249, 203, 233, 190, 60, 176, 13, 34, 63, 143, 206, 107, 48, 196, 186, 76, 163, 154, 87, 236, 0, 211, 215, 110, 100, 180, 101, 202, 179, 150, 55, 196, 64, 17, 183, 128, 254, 94, 19, 141, 191, 161, 26, 131, 139, 229, 202, 105, 232, 62, 70, 44, 123, 198, 163, 142, 3, 154, 166, 204, 228, 9, 101, 145, 196, 43, 171, 240, 233, 132, 221, 86, 66, 45, 43, 58, 152, 172, 76, 63, 209, 49, 51, 119, 143, 131, 

In [15]:
arr = np.reshape(SRAM_DATA, (1156, 40))
print(list(arr[1155]))
print(list(arr[1154]))
print(list(arr[1155-288]))

[86, 35, 187, 182, 48, 44, 17, 10, 40, 219, 146, 167, 126, 74, 204, 79, 26, 222, 105, 36, 84, 6, 109, 3, 133, 55, 202, 19, 27, 248, 109, 48, 123, 198, 44, 141, 13, 0, 0, 0]
[197, 13, 53, 183, 194, 168, 89, 1, 14, 164, 94, 207, 206, 14, 5, 60, 41, 249, 225, 184, 31, 131, 206, 5, 87, 200, 169, 26, 207, 159, 171, 198, 113, 87, 99, 142, 220, 0, 0, 0]
[119, 112, 134, 120, 121, 36, 237, 33, 15, 7, 218, 237, 186, 201, 202, 218, 112, 52, 102, 19, 125, 202, 116, 178, 125, 151, 108, 164, 178, 203, 226, 138, 177, 232, 94, 30, 70, 0, 0, 0]


In [16]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)


Default Signals Set
[128]


In [17]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0